# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 1: КОНФИГУРАЦИЯ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import userdata
import torch

# Токен из Colab Secrets (добавь HF_TOKEN в левую панель 🔑)
HF_TOKEN = userdata.get('HF_TOKEN')  # или впиши строкой: '...'

CONFIG = {
    'hf_token':     HF_TOKEN,
    'hf_dataset':   'AvaSiG/argos-dataset',
    'hf_model_out': 'AvaSiG/argos-mistral-12b',
    'train_file':   'argos_train_v2.jsonl',
    'val_file':     'argos_val_v2.jsonl',
    'base_model':   'unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit',
    'max_seq_len':  2048,
    'lora_r':       32,
    'lora_alpha':   64,
    'lora_dropout': 0.05,
    'epochs':       3,
    'batch_size':   4,
    'grad_accum':   4,
    'lr':           2e-4,
    'warmup_ratio': 0.05,
    'gguf_quant':   'q4_k_m',
    'push_to_hub':  True,
}

print('✅ Config OK | Token:', HF_TOKEN[:15]+'...' if HF_TOKEN else '❌ НЕТ ТОКЕНА!')
print(f'   Модель:  {CONFIG["base_model"]}')
print(f'   Выход:   {CONFIG["hf_model_out"]}')


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 1: КОНФИГУРАЦИЯ (настроить здесь)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from google.colab import userdata

CONFIG = {
    # HuggingFace
    'hf_token':       userdata.get('HF_TOKEN'),
    'hf_dataset':     'AvaSiG/argos-dataset',
    'hf_model_out':   'AvaSiG/argos-mistral-12b',  # куда сохранить
    'train_file':     'argos_train_v2.jsonl',
    'val_file':       'argos_val_v2.jsonl',
    
    # Базовая модель
    'base_model':     'unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit',
    'max_seq_len':    2048,
    
    # LoRA
    'lora_r':         32,     # rank (32 для A100 хватает памяти)
    'lora_alpha':     64,
    'lora_dropout':   0.05,
    
    # Обучение
    'epochs':         3,
    'batch_size':     4,      # A100 40GB: 4-8
    'grad_accum':     4,      # effective batch = 16
    'lr':             2e-4,
    'warmup_ratio':   0.05,
    
    # Экспорт для V100
    'gguf_quant':     'q4_k_m',  # 4-bit для V100 16GB
    'push_to_hub':    True,
}

print('✅ Конфигурация загружена')
print(f'   Модель:  {CONFIG["base_model"]}')
print(f'   Выход:   {CONFIG["hf_model_out"]}')
print(f'   Эпох:    {CONFIG["epochs"]}')
print(f'   Batch:   {CONFIG["batch_size"]} × {CONFIG["grad_accum"]} = {CONFIG["batch_size"]*CONFIG["grad_accum"]} eff.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 2: УСТАНОВКА (один раз)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('unsloth[colab-new]')
pip('trl', 'peft', 'accelerate', 'bitsandbytes')
pip('huggingface_hub', 'datasets')

print('✅ Зависимости установлены')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 3: GPU ИНФОРМАЦИЯ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

gpu = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1024**3
print(f'GPU:  {gpu.name}')
print(f'VRAM: {vram:.1f} GB')
print(f'BF16: {torch.cuda.is_bf16_supported()}')

# Авто-настройка batch_size по VRAM
if vram >= 75:  # A100 80GB
    CONFIG['batch_size'] = 8
    CONFIG['grad_accum'] = 2
elif vram >= 38:  # A100 40GB
    CONFIG['batch_size'] = 4
    CONFIG['grad_accum'] = 4
elif vram >= 15:  # V100 16GB
    CONFIG['batch_size'] = 2
    CONFIG['grad_accum'] = 8

print(f'\nАвто batch: {CONFIG["batch_size"]} × {CONFIG["grad_accum"]} = {CONFIG["batch_size"]*CONFIG["grad_accum"]} eff.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 4: ЗАГРУЗКА МОДЕЛИ + LoRA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['base_model'],
    max_seq_length=CONFIG['max_seq_len'],
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,
    token=CONFIG['hf_token'],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    use_rslora=True,
    random_state=42,
)

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Модель загружена | Trainable: {params/1e6:.1f}M params | LoRA r={CONFIG["lora_r"]}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 5: ДАТАСЕТ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='mistral')

def format_examples(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

ds_train = load_dataset(
    CONFIG['hf_dataset'],
    data_files=CONFIG['train_file'],
    split='train',
    token=CONFIG['hf_token']
).map(format_examples, batched=True, remove_columns=['messages'])

ds_val = load_dataset(
    CONFIG['hf_dataset'],
    data_files=CONFIG['val_file'],
    split='train',
    token=CONFIG['hf_token']
).map(format_examples, batched=True, remove_columns=['messages'])

print(f'✅ Датасет загружен: {len(ds_train)} train | {len(ds_val)} val')
print(f'   Пример ({len(ds_train[0]["text"])} chars):', ds_train[0]['text'][:120], '...')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 6: CALLBACK — АВТО-СОХРАНЕНИЕ НА HF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from transformers import TrainerCallback
from huggingface_hub import HfApi

class AutoPushCallback(TrainerCallback):
    """Авто-пуш на HuggingFace после каждой эпохи."""
    def __init__(self, model, tokenizer, repo_id, token):
        self.model = model
        self.tokenizer = tokenizer
        self.repo_id = repo_id
        self.token = token
        self.api = HfApi(token=token)
        # Создать репо если нет
        try:
            self.api.create_repo(repo_id, repo_type='model', private=True, exist_ok=True)
            print(f'📦 HF репо: https://hf.co/{repo_id}')
        except Exception as e:
            print(f'Репо: {e}')
    
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        print(f'\n💾 Эпоха {epoch} завершена. Сохраняю на HF...')
        try:
            self.model.push_to_hub(
                self.repo_id,
                token=self.token,
                commit_message=f'Epoch {epoch} | loss={state.log_history[-1].get("loss","?"):.4f}'
            )
            self.tokenizer.push_to_hub(self.repo_id, token=self.token)
            print(f'✅ Эпоха {epoch} → HF: https://hf.co/{self.repo_id}')
        except Exception as e:
            print(f'❌ Ошибка пуша: {e}')

print('✅ AutoPush callback готов')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 7: ЗАПУСК ОБУЧЕНИЯ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from trl import SFTTrainer
from transformers import TrainingArguments
import time

auto_push = AutoPushCallback(
    model, tokenizer,
    repo_id=CONFIG['hf_model_out'],
    token=CONFIG['hf_token']
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_len'],
    dataset_num_proc=4,
    packing=True,  # макс. эффективность
    args=TrainingArguments(
        output_dir='./outputs',
        num_train_epochs=CONFIG['epochs'],
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['grad_accum'],
        learning_rate=CONFIG['lr'],
        warmup_ratio=CONFIG['warmup_ratio'],
        lr_scheduler_type='cosine',
        optim='adamw_8bit',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        weight_decay=0.01,
        max_grad_norm=1.0,
        dataloader_num_workers=4,
        report_to='none',
        seed=42,
    ),
    callbacks=[auto_push],
)

print(f'🚀 Запуск обучения ({CONFIG["epochs"]} эпохи)...')
t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0

print(f'\n✅ Обучение завершено за {elapsed/60:.1f} мин')
print(f'   Final loss: {result.training_loss:.4f}')
print(f'   Samples/sec: {len(ds_train)*CONFIG["epochs"]/elapsed:.1f}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 8: ЭКСПОРТ GGUF ДЛЯ V100 + ФИНАЛЬНЫЙ ПУШ НА HF
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, shutil

# Сохранить LoRA адаптер
model.save_pretrained('argos-lora-final')
tokenizer.save_pretrained('argos-lora-final')
print('✅ LoRA адаптер сохранён: ./argos-lora-final/')

# Экспорт GGUF Q4_K_M (оптимально для V100 16GB)
print(f'\n⚙️  Экспорт GGUF {CONFIG["gguf_quant"]}...')
model.save_pretrained_gguf(
    'argos-gguf',
    tokenizer,
    quantization_method=CONFIG['gguf_quant']
)
print('✅ GGUF сохранён: ./argos-gguf/')

# Создать Modelfile для Ollama
modelfile = '''FROM ./argos-mistral-12b.Q4_K_M.gguf

SYSTEM """Ты — АРГОС (Argos Universal OS), автономная AI-система Всеволода.
Отвечай по-русски, кратко и точно. ВЫПОЛНЯЙ задачи — не описывай процесс."""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
'''
with open('argos-gguf/Modelfile', 'w') as f:
    f.write(modelfile)
print('✅ Modelfile создан')

# Финальный пуш на HF (LoRA + GGUF)
if CONFIG['push_to_hub']:
    from huggingface_hub import HfApi
    api = HfApi(token=CONFIG['hf_token'])
    
    print('\n📤 Финальный пуш на HuggingFace...')
    
    # LoRA адаптер
    api.upload_folder(
        folder_path='argos-lora-final',
        repo_id=CONFIG['hf_model_out'],
        repo_type='model',
        commit_message='Final LoRA adapter + GGUF Q4_K_M for V100'
    )
    
    # GGUF файл
    import glob
    for gguf in glob.glob('argos-gguf/*.gguf'):
        print(f'   Загружаю {gguf}...')
        api.upload_file(
            path_or_fileobj=gguf,
            path_in_repo=os.path.basename(gguf),
            repo_id=CONFIG['hf_model_out'],
            repo_type='model'
        )
    
    # Modelfile
    api.upload_file(
        path_or_fileobj='argos-gguf/Modelfile',
        path_in_repo='Modelfile',
        repo_id=CONFIG['hf_model_out'],
        repo_type='model'
    )
    
    print(f'✅ Всё загружено: https://hf.co/{CONFIG["hf_model_out"]}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЯЧЕЙКА 9: СКАЧАТЬ ФАЙЛЫ + ИНСТРУКЦИЯ ДЛЯ V100
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import shutil
from google.colab import files

# Создать архив для скачивания
shutil.make_archive('/content/argos_adapter', 'zip', 'argos-lora-final')
shutil.make_archive('/content/argos_gguf_v100', 'zip', 'argos-gguf')

print('📦 Архивы готовы:')
print('   /content/argos_adapter.zip — LoRA адаптер')
print('   /content/argos_gguf_v100.zip — GGUF для V100 + Modelfile')

# Авто-скачать
files.download('/content/argos_gguf_v100.zip')

print('\n' + '='*60)
print('УСТАНОВКА НА V100 (после скачивания):')
print('='*60)
print('''
# 1. Распаковать
unzip argos_gguf_v100.zip -d ~/argos-model/

# 2. Создать модель в Ollama
cd ~/argos-model/
ollama create argos-v2 -f Modelfile

# 3. Тест
ollama run argos-v2 "Кто ты?"

# 4. В .env ARGOS добавить:
# OLLAMA_MODEL=argos-v2
# OLLAMA_FAST_MODEL=argos-v2

# 5. Перезапустить ARGOS
# python main.py --no-gui --dashboard
''')
print(f'\n🎉 Модель доступна: https://hf.co/{CONFIG["hf_model_out"]}')